In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
os.environ['MUJOCO_GL'] = 'egl'

import functools

import jax.numpy as jp
import numpy as np
import jax

from mujoco_playground import registry
from mujoco_playground.config import locomotion_params
from mujoco_playground._src.locomotion.go2 import go2_constants as consts

from brax.training.agents.apg import networks as apg_networks

from brax.envs.wrappers import training as brax_training


from brax.io import model

env_name = "Go2Joystick2"
env_cfg = registry.get_default_config(env_name)
randomizer = registry.get_domain_randomizer(env_name)

Warp DeprecationWarning: The namespace `warp.context` will soon be removed from the public API. It can still be accessed from `warp._src.context` but might be changed or removed without notice.
Warp DeprecationWarning: The symbol `warp.context.Module` will soon be removed from the public API. Use `warp.Module` instead.
Warp DeprecationWarning: The symbol `warp.context.get_module` will soon be removed from the public API. Use `warp.get_module` instead.
Warp DeprecationWarning: The namespace `warp.math` will soon be removed from the public API. It can still be accessed from `warp._src.math` but might be changed or removed without notice.


# Load params

In [10]:
params_path = "/data/mujoco_playground/mujoco_playground/experimental/learning/checkpoints/Go2Joystick2-20260225-020956-apg/params.pkl"
obs_dim = consts.RESIDUAL_OBS_DIM
act_dim = consts.RESIDUAL_ACT_DIM
onnx_path = "/data/mujoco_playground/mujoco_playground/experimental/sim2sim/onnx/go2_apg2_residual_policy.onnx"

# params_path = consts.ANCHOR_PATH
# obs_dim = consts.ANCHOR_OBS_DIM
# act_dim = consts.ANCHOR_ACT_DIM
# onnx_path = "/data/mujoco_playground/mujoco_playground/experimental/sim2sim/onnx/go2_apg2_anchor_policy.onnx"

In [6]:
apg_params = locomotion_params.brax_apg_config(env_name)
params = model.load_params(params_path)
params = (params["normalizer_params"], params["policy_params"])
print(params[0].mean['state'].shape, params[0].std['state'].shape)

(48,) (48,)


# Brax to torch

In [7]:
import torch
import torch.nn as nn
import numpy as np
import jax
import jax.numpy as jnp
import flax
from flax import linen


# ================================================================
# 🧩 1. Go2Policy 定义
# ================================================================
class Go2Policy(nn.Module):
    def __init__(self, obs_dim=40, act_dim=12, obs_mean=None, obs_std=None, eps=1e-10):
        super().__init__()
        self.eps = eps

        # 注册归一化参数
        if obs_mean is not None and obs_std is not None:
            self.register_buffer("obs_mean", torch.tensor(obs_mean, dtype=torch.float32))
            self.register_buffer("obs_std", torch.tensor(obs_std, dtype=torch.float32))
        else:
            self.register_buffer("obs_mean", torch.zeros(obs_dim))
            self.register_buffer("obs_std", torch.ones(obs_dim))

        # 网络结构：Linear → ELU → LayerNorm → Linear → ELU → LayerNorm → Linear
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 256),
            nn.ELU(),
            nn.LayerNorm(256, eps=1e-6),
            nn.Linear(256, 128),
            nn.ELU(),
            nn.LayerNorm(128, eps=1e-6),
            nn.Linear(128, act_dim * 2),
        )

    # 模仿 running_statistics.normalize
    def normalize_obs(self, x):
        return (x - self.obs_mean) / (self.obs_std + self.eps)

    # 前向传播
    def forward(self, x):
        x = self.normalize_obs(x)
        out = self.net(x)

        mean, log_std = out.chunk(2, dim=-1)
        std = torch.exp(log_std)
        mean_tanh = torch.tanh(mean)
        return mean_tanh, std


# ================================================================
# ⚙️ 2. 权重加载：JAX → PyTorch
# ================================================================
def load_jax_to_torch(jax_params, torch_model):
    with torch.no_grad():
        # hidden_0
        torch_model.net[0].weight.data = torch.tensor(
            np.array(jax_params["hidden_0"]["kernel"]).T, dtype=torch.float32
        )
        torch_model.net[0].bias.data = torch.tensor(
            np.array(jax_params["hidden_0"]["bias"]), dtype=torch.float32
        )
        torch_model.net[2].weight.data = torch.tensor(
            np.array(jax_params["LayerNorm_0"]["scale"]), dtype=torch.float32
        )
        torch_model.net[2].bias.data = torch.tensor(
            np.array(jax_params["LayerNorm_0"]["bias"]), dtype=torch.float32
        )

        # hidden_1
        torch_model.net[3].weight.data = torch.tensor(
            np.array(jax_params["hidden_1"]["kernel"]).T, dtype=torch.float32
        )
        torch_model.net[3].bias.data = torch.tensor(
            np.array(jax_params["hidden_1"]["bias"]), dtype=torch.float32
        )
        torch_model.net[5].weight.data = torch.tensor(
            np.array(jax_params["LayerNorm_1"]["scale"]), dtype=torch.float32
        )
        torch_model.net[5].bias.data = torch.tensor(
            np.array(jax_params["LayerNorm_1"]["bias"]), dtype=torch.float32
        )

        # hidden_2
        torch_model.net[6].weight.data = torch.tensor(
            np.array(jax_params["hidden_2"]["kernel"]).T, dtype=torch.float32
        )
        torch_model.net[6].bias.data = torch.tensor(
            np.array(jax_params["hidden_2"]["bias"]), dtype=torch.float32
        )

    print("✅ JAX 权重已成功复制到 PyTorch Sequential 模型！")


# ================================================================
# 🔍 3. 分层比较：精确查看哪一层开始不同
# ================================================================
def compare_layers(jax_params, torch_model, obs):
    """
    分层比较 JAX 与 PyTorch 的输出，逐层打印差异
    """
    x_jax = obs
    x_torch = torch.tensor(np.array(obs), dtype=torch.float32)

    # 取出 JAX 的权重层（按顺序）
    jax_layers = [
        jax_params["hidden_0"],
        jax_params["hidden_1"],
        jax_params["hidden_2"],
    ]

    layer_idx = 0
    for i, module in enumerate(torch_model.net):
        if isinstance(module, nn.Linear):
            # 对应 JAX Dense
            W = np.array(jax_layers[layer_idx]["kernel"])
            b = np.array(jax_layers[layer_idx]["bias"])
            x_jax = jnp.dot(x_jax, W) + b
            x_torch = module(x_torch)
            diff = np.max(np.abs(np.array(x_jax) - x_torch.detach().numpy()))
            print(f"[Linear {layer_idx}] max diff = {diff:.6f}")
            layer_idx += 1

        elif isinstance(module, nn.LayerNorm):
            # 对应 LayerNorm
            scale = np.array(jax_params[f"LayerNorm_{layer_idx-1}"]["scale"])
            bias = np.array(jax_params[f"LayerNorm_{layer_idx-1}"]["bias"])

            mean = np.mean(np.array(x_jax), axis=-1, keepdims=True)
            var = np.var(np.array(x_jax), axis=-1, keepdims=True)
            x_jax = (np.array(x_jax) - mean) / np.sqrt(var + 1e-6)
            x_jax = scale * x_jax + bias

            x_torch = module(x_torch)
            diff = np.max(np.abs(np.array(x_jax) - x_torch.detach().numpy()))
            print(f"[LayerNorm {layer_idx-1}] max diff = {diff:.6f}")

        elif isinstance(module, nn.ELU):
            x_jax = linen.elu(x_jax)
            x_torch = module(x_torch)
            diff = np.max(np.abs(np.array(x_jax) - x_torch.detach().numpy()))
            print(f"[ELU {layer_idx-1}] max diff = {diff:.6f}")


In [8]:
obs_mean = np.array(params[0].mean['state'])
obs_std = np.array(params[0].std['state'])
torch_model = Go2Policy(
    obs_dim=obs_dim, 
    act_dim=act_dim,
    obs_mean=obs_mean,
    obs_std=obs_std
)
load_jax_to_torch(params[1]['params'], torch_model)

✅ JAX 权重已成功复制到 PyTorch Sequential 模型！


# Test

In [9]:
# 层级对比
obs_rand = np.random.randn(1, obs_dim).astype(np.float32)
compare_layers(params[1]['params'], torch_model, obs_rand)


[Linear 0] max diff = 0.000000
[ELU 0] max diff = 0.000000
[LayerNorm 0] max diff = 0.000001
[Linear 1] max diff = 0.000000
[ELU 1] max diff = 0.000000
[LayerNorm 1] max diff = 0.000001
[Linear 2] max diff = 0.000000


# Torch to onnx

In [11]:
torch_model.eval()
torch.onnx.export(
    torch_model,
    torch.zeros((1, obs_dim), dtype=torch.float32),
    onnx_path,
    input_names=["obs"],
    output_names=["actions", "std"],
    dynamic_axes={
        "obs": {0: "batch_size"},
        "actions": {0: "batch_size"},
        "std": {0: "batch_size"},
    },
    opset_version=17,
    dynamo=False
)

/tmp/ipykernel_751883/740643528.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
